In [ ]:
# Production-Grade Spark Data Skew Lab (Local Spark 3.5)
'''This lab is designed specifically for:

Local Spark 3.5
Observable skew in Spark UI
Long-running stages (minutes)
GB-scale shuffle
Clear skew metrics
Real interview-style investigation

This version intentionally creates:

Extreme reducer imbalance

so you can visually observe:

Long-tail tasks
Huge shuffle reads
Uneven executor workload
Spill
GC pressure'''

In [1]:
#STEP 1 — Create Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StableSkewLab") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.memory.fraction", "0.6") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [7]:
# STEP 2 — Create Controlled Skew

from pyspark.sql.functions import *

N = 8_000_000

large_df = spark.range(0, N) \
    .withColumn(
        "key",
        when(col("id") < 7_700_000, lit(1))
        .otherwise(col("id"))
    ) \
    .withColumn(
        "payload",
        repeat(lit("X"), 50)
    )

In [8]:
# STEP 3 — FORCE REAL SKEW
skew_df = large_df.repartition(8, "key")

In [6]:
# STEP 4 — VERIFY PARTITION IMBALANCE
from pyspark.sql.functions import spark_partition_id

skew_df \
    .withColumn("pid", spark_partition_id()) \
    .groupBy("pid") \
    .count() \
    .orderBy("pid") \
    .show(truncate=False)

+---+-------+
|pid|count  |
+---+-------+
|0  |37067  |
|1  |37581  |
|2  |37250  |
|3  |37671  |
|4  |37736  |
|5  |7737615|
|6  |37621  |
|7  |37459  |
+---+-------+



In [9]:
# STEP 5 — Create Small Table
small_df = spark.range(0, 1000) \
    .withColumnRenamed("id", "key")

small_df = small_df.repartition(8, "key")

In [10]:
# STEP 6 — FORCE SHUFFLE JOIN
joined_df = skew_df.join(
    small_df,
    "key"
)

In [11]:
# VERIFY PLAN --> SortMergeJoin , Exchange
joined_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [key])
:- RepartitionByExpression [key#241L], 8
:  +- Project [id#239L, key#241L, repeat(X, 50) AS payload#244]
:     +- Project [id#239L, CASE WHEN (id#239L < cast(7700000 as bigint)) THEN cast(1 as bigint) ELSE id#239L END AS key#241L]
:        +- Range (0, 8000000, step=1, splits=Some(8))
+- RepartitionByExpression [key#250L], 8
   +- Project [id#248L AS key#250L]
      +- Range (0, 1000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
key: bigint, id: bigint, payload: string
Project [key#241L, id#239L, payload#244]
+- Join Inner, (key#241L = key#250L)
   :- RepartitionByExpression [key#241L], 8
   :  +- Project [id#239L, key#241L, repeat(X, 50) AS payload#244]
   :     +- Project [id#239L, CASE WHEN (id#239L < cast(7700000 as bigint)) THEN cast(1 as bigint) ELSE id#239L END AS key#241L]
   :        +- Range (0, 8000000, step=1, splits=Some(8))
   +- RepartitionByExpression [key#250L], 8
      +- Project [id#248L AS key#250L]
   

In [12]:
# STEP 7 — EXECUTE LONG-RUNNING ACTION ---> THIS creates visible skew.
joined_df.groupBy("key") \
    .count() \
    .write.mode("overwrite") \
    .parquet("skew_output")

In [13]:
# VISUALIZE AGGREGATION SKEW
agg_df = skew_df.groupBy("key").count()

agg_df.write.mode("overwrite").parquet("agg_output")

In [14]:
# CREATE JOIN SKEW --> Join skew is MUCH more realistic in production.
#SMALL DIMENSION TABLE 
small_df = spark.range(0, 1000) \
    .withColumnRenamed("id", "key")

small_df = small_df.repartition(8, "key")


In [15]:
# FORCE SHUFFLE JOIN
join_df = skew_df.join(
    small_df,
    "key"
)

In [16]:
# VERIFY EXECUTION PLAN --> You MUST see:  SortMergeJoin , Exchange hashpartitioning
join_df.groupBy("key") \
    .count() \
    .write.mode("overwrite") \
    .parquet("join_output")

In [17]:
# DETECTING SKEW PROPERLY
# METHOD 1 — Partition Counts
skew_df \
    .withColumn("pid", spark_partition_id()) \
    .groupBy("pid") \
    .count() \
    .orderBy(desc("count")) \
    .show()

+---+-------+
|pid|  count|
+---+-------+
|  5|7737615|
|  4|  37736|
|  3|  37671|
|  6|  37621|
|  1|  37581|
|  7|  37459|
|  2|  37250|
|  0|  37067|
+---+-------+



In [18]:
# METHOD 2 — Key Frequency
skew_df.groupBy("key") \
    .count() \
    .orderBy(desc("count")) \
    .show()

+-------+-------+
|    key|  count|
+-------+-------+
|      1|7700000|
|7700009|      1|
|7700002|      1|
|7700005|      1|
|7700024|      1|
|7700000|      1|
|7700039|      1|
|7700018|      1|
|7700043|      1|
|7700003|      1|
|7700048|      1|
|7700020|      1|
|7700050|      1|
|7700028|      1|
|7700076|      1|
|7700022|      1|
|7700085|      1|
|7700011|      1|
|7700087|      1|
|7700023|      1|
+-------+-------+
only showing top 20 rows



In [ ]:
# METHOD 3 — Spark UI
'''
Look for:

long-running tasks
huge shuffle read
spill
skewed reducers
'''

In [19]:
# METHOD 4 — Explain Plan --> Look for: Exchange , SortMergeJoin HashAggregate These cause shuffle.
join_df.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [key])
:- RepartitionByExpression [key#241L], 8
:  +- Project [id#239L, key#241L, repeat(X, 50) AS payload#244]
:     +- Project [id#239L, CASE WHEN (id#239L < cast(7700000 as bigint)) THEN cast(1 as bigint) ELSE id#239L END AS key#241L]
:        +- Range (0, 8000000, step=1, splits=Some(8))
+- RepartitionByExpression [key#279L], 8
   +- Project [id#277L AS key#279L]
      +- Range (0, 1000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
key: bigint, id: bigint, payload: string
Project [key#241L, id#239L, payload#244]
+- Join Inner, (key#241L = key#279L)
   :- RepartitionByExpression [key#241L], 8
   :  +- Project [id#239L, key#241L, repeat(X, 50) AS payload#244]
   :     +- Project [id#239L, CASE WHEN (id#239L < cast(7700000 as bigint)) THEN cast(1 as bigint) ELSE id#239L END AS key#241L]
   :        +- Range (0, 8000000, step=1, splits=Some(8))
   +- RepartitionByExpression [key#279L], 8
      +- Project [id#277L AS key#279L]
   

In [20]:
# SOLVE USING BROADCAST JOIN
# WHY BROADCAST WORKS - Instead of shuffling BOTH tables: 
# Small table copied to executors  Large table stays in place.


In [21]:
# ENABLE BROADCAST
from pyspark.sql.functions import broadcast

broadcast_join = skew_df.join(
    broadcast(small_df),
    "key"
)

In [22]:
# VERIFY PLAN --> You should see:  BroadcastHashJoin
broadcast_join.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [key])
:- RepartitionByExpression [key#241L], 8
:  +- Project [id#239L, key#241L, repeat(X, 50) AS payload#244]
:     +- Project [id#239L, CASE WHEN (id#239L < cast(7700000 as bigint)) THEN cast(1 as bigint) ELSE id#239L END AS key#241L]
:        +- Range (0, 8000000, step=1, splits=Some(8))
+- ResolvedHint (strategy=broadcast)
   +- RepartitionByExpression [key#279L], 8
      +- Project [id#277L AS key#279L]
         +- Range (0, 1000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
key: bigint, id: bigint, payload: string
Project [key#241L, id#239L, payload#244]
+- Join Inner, (key#241L = key#279L)
   :- RepartitionByExpression [key#241L], 8
   :  +- Project [id#239L, key#241L, repeat(X, 50) AS payload#244]
   :     +- Project [id#239L, CASE WHEN (id#239L < cast(7700000 as bigint)) THEN cast(1 as bigint) ELSE id#239L END AS key#241L]
   :        +- Range (0, 8000000, step=1, splits=Some(8))
   +- ResolvedHint (strategy=broadcast)


In [23]:
# EXECUTE
broadcast_join.groupBy("key") \
    .count() \
    .write.mode("overwrite") \
    .parquet("broadcast_output")

In [24]:
# SOLVE USING AQE
# ENABLE AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")

spark.conf.set(
    "spark.sql.adaptive.skewJoin.enabled",
    "true"
)

In [25]:
# RUN AGAIN
aqe_join = skew_df.join(
    small_df,
    "key"
)

aqe_join.groupBy("key") \
    .count() \
    .write.mode("overwrite") \
    .parquet("aqe_output")

In [ ]:
# SPARK UI OBSERVATION --> You may observe: AdaptiveSparkPlan , CustomShuffleReader ,SplitSkewedPartition



In [26]:
# SOLVE USING SALTING
# WHY SALTING WORKS Instead of: All key=1 rows -> same reducer Salt distributes workload.
# ADD SALT
salted_large = skew_df.withColumn(
    "salt",
    floor(rand() * 10)
)

In [29]:
salted_large.show(5,False)

+-------+-------+--------------------------------------------------+----+
|id     |key    |payload                                           |salt|
+-------+-------+--------------------------------------------------+----+
|7700011|7700011|XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|1   |
|7700012|7700012|XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|7   |
|7700019|7700019|XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|4   |
|7700021|7700021|XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|2   |
|7700053|7700053|XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX|5   |
+-------+-------+--------------------------------------------------+----+
only showing top 5 rows



In [27]:
# EXPAND SMALL TABLE
from pyspark.sql.functions import explode, array

salt_values = array(*[lit(i) for i in range(10)])

salted_small = small_df.withColumn(
    "salt",
    explode(salt_values)
)

In [30]:
salted_small.show(5,False)

+---+----+
|key|salt|
+---+----+
|2  |0   |
|2  |1   |
|2  |2   |
|2  |3   |
|2  |4   |
+---+----+
only showing top 5 rows



In [28]:
# SALTED JOIN
salted_join = salted_large.join(
    salted_small,
    ["key", "salt"]
)

In [31]:
# EXECUTE
salted_join.groupBy("key") \
    .count() \
    .write.mode("overwrite") \
    .parquet("salted_output")

In [ ]:
# MOST IMPORTANT SPARK UI SCREENS
'''
STAGES TAB

Observe:
long-running tasks
stragglers
skewed reducers

TASKS TAB

Sort by:
Duration
Shuffle Read
Input Size

SQL TAB

Observe:
Exchange
SortMergeJoin
BroadcastHashJoin
AdaptiveSparkPlan


'''

In [ ]:
# REAL PRODUCTION INSIGHTS
'''
Aggregation Skew

Common in:
groupBy(customer_id)
when few customers dominate traffic.


Join Skew

Common in:
country='US'
status='ACTIVE'

Highly repeated keys

AQE

Excellent for:
moderate skew
dynamic optimization

Salting

Used when:

AQE insufficient
huge skew exists
large-large joins

Broadcast

Best when:
One table is small

'''

In [ ]:
'''
MOST IMPORTANT LESSON

Most Spark performance issues are fundamentally:

Shuffle + Skew + Spill

NOT:

CPU computation

That is the core production insight behind Spark data skew.
'''